In [5]:
import torch
import torchvision
import cv2
import numpy as np
import os
import glob
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

# --- UPDATED CALIBRATION METRICS ---
PIXELS_PER_MM = 10.157126168224298
CARD_WIDTH_MM = 85.60
CARD_HEIGHT_MM = 53.98
TRUE_DIM_LONG = 113.0
TRUE_DIM_SHORT = 62.0

CAMERA_MATRIX = np.array([
    [6.28627483e+03, 0.00000000e+00, 1.04902659e+03],
    [0.00000000e+00, 5.95750482e+03, 2.69757463e+03],
    [0.00000000e+00, 0.00000000e+00, 1.00000000e+00]
], dtype=np.float32)

DISTORTION_COEFFICIENTS = np.array([
    [-0.27477453,  2.58453069,  0.01454981,  0.01631264, -3.94001836]
], dtype=np.float32)

# --- DIRECTORIES ---
TRAIN_IMAGES_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox/train"
TEST_IMAGES_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox/test"
WEIGHTS_PATH = "/kaggle/input/models/muhammadunssrahim/tea-box-detector/pytorch/default/1/teabox_mask_rcnn.pth"
OUTPUT_DIR = "/kaggle/working/outputfolder_revised"
# -----------------------------------

def load_segmentation_model(weights_path, device):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, 2)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def correct_lens_distortion(img, mtx, dist):
    h, w = img.shape[:2]
    new_camera_mtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 1, (w, h))
    dst = cv2.undistort(img, mtx, dist, None, new_camera_mtx)
    x, y, roi_w, roi_h = roi
    return dst[y:y+roi_h, x:x+roi_w]

def measure_baseline_card(img, ppm_ratio):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([80, 70, 50]), np.array([105, 255, 255]))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_contours = [c for c in contours if cv2.contourArea(c) > 500]
    if not valid_contours:
        return None
    largest_contour = max(valid_contours, key=cv2.contourArea)
    _, _, pixel_w, pixel_h = cv2.boundingRect(largest_contour)
    return pixel_w / ppm_ratio, pixel_h / ppm_ratio

def measure_target_ai(img, model, device, ppm_ratio):
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor_img = torchvision.transforms.functional.to_tensor(rgb_img).unsqueeze(0).to(device)
    with torch.no_grad():
        prediction = model(tensor_img)[0]
    scores = prediction['scores'].cpu().numpy()
    if len(scores) == 0 or scores[0] < 0.15:
        return None
    mask = (prediction['masks'][0, 0].cpu().numpy() > 0.5).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest_contour = max(contours, key=cv2.contourArea)
    _, _, pixel_w, pixel_h = cv2.boundingRect(largest_contour)
    
    # Return the largest contour as well so we can draw it later
    return pixel_w / ppm_ratio, pixel_h / ppm_ratio, scores[0], largest_contour

def run_eval_pipeline(train_images_dir, test_images_dir, weights_path, output_dir):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    model = load_segmentation_model(weights_path, device)
    
    train_paths = sorted(glob.glob(f"{train_images_dir}/*.jpg"))[:3]
    test_paths = sorted(glob.glob(f"{test_images_dir}/*.jpg"))
    image_paths = train_paths + test_paths
    
    all_results = []
    
    print(f"Launching evaluation loop over {len(image_paths)} total files (3 Train + Test)...\n")
    
    for path in image_paths:
        filename = os.path.basename(path)
        source_folder = "TRAIN" if path in train_paths else "TEST"
        raw_img = cv2.imread(path)
        if raw_img is None:
            continue
            
        img = correct_lens_distortion(raw_img, CAMERA_MATRIX, DISTORTION_COEFFICIENTS)
        # Create a copy to draw on
        annotated_img = img.copy() 
        
        card_dims = measure_baseline_card(img, PIXELS_PER_MM)
        active_ppm = PIXELS_PER_MM
        
        if card_dims:
            c_w, _ = card_dims
            pixel_width_of_card = c_w * PIXELS_PER_MM
            active_ppm = pixel_width_of_card / CARD_WIDTH_MM
            
        box_result = measure_target_ai(img, model, device, active_ppm)
        
        if box_result:
            b_w, b_h, conf, contour = box_result
            pred_short, pred_long = sorted([b_w, b_h])
            
            err_long = abs(pred_long - TRUE_DIM_LONG)
            err_short = abs(pred_short - TRUE_DIM_SHORT)
            
            pct_long = (err_long / TRUE_DIM_LONG) * 100
            pct_short = (err_short / TRUE_DIM_SHORT) * 100
            avg_pct_err = (pct_long + pct_short) / 2.0
            
            all_results.append({
                'filename': filename,
                'source': source_folder,
                'pred_long': pred_long,
                'pred_short': pred_short,
                'pct_long': pct_long,
                'pct_short': pct_short,
                'avg_error': avg_pct_err
            })
            
            # --- DRAWING ON THE IMAGE ---
            # 1. Draw Yellow Polygon Mask
            overlay = annotated_img.copy()
            # BGR for Yellow is (0, 255, 255)
            cv2.drawContours(overlay, [contour], -1, (0, 255, 255), -1) 
            # Apply alpha blending for semi-transparent mask fill
            cv2.addWeighted(overlay, 0.4, annotated_img, 0.6, 0, annotated_img)
            # Draw solid yellow border
            cv2.drawContours(annotated_img, [contour], -1, (0, 255, 255), 3) 

            # 2. Add Text Metrics
            text_lines = [
                f"Source: {source_folder}",
                f"Conf: {conf:.2f}",
                f"Dims: {pred_long:.1f} x {pred_short:.1f} mm",
                f"Err(L): {err_long:.1f}mm ({pct_long:.1f}%)",
                f"Err(S): {err_short:.1f}mm ({pct_short:.1f}%)",
                f"Avg Err: {avg_pct_err:.1f}%"
            ]
            
            y_offset = 60
            for line in text_lines:
                # Adding black background stroke for readability on light/dark backgrounds
                cv2.putText(annotated_img, line, (40, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 5, cv2.LINE_AA)
                cv2.putText(annotated_img, line, (40, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2, cv2.LINE_AA)
                y_offset += 45

            print(f"[{source_folder}][{filename}] Conf: {conf:.4f}")
            print(f"   -> Long Dim (113mm): {pred_long:.2f}mm | Abs: {err_long:.2f}mm | Pct: {pct_long:.2f}%")
            print(f"   -> Short Dim (62mm): {pred_short:.2f}mm | Abs: {err_short:.2f}mm | Pct: {pct_short:.2f}%\n")
        else:
            cv2.putText(annotated_img, "Target localization breach", (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
            print(f"[{source_folder}][{filename}] Target localization breach - skipping metrics.\n")
            
        # Save the finalized image with drawn annotations
        output_filepath = os.path.join(output_dir, filename)
        cv2.imwrite(output_filepath, annotated_img)
            
    if all_results:
        print("=" * 60)
        print(f"FILES SAVED TO: {output_dir}")
        print("=" * 60)
        print("PERFORMANCE RANKING (BEST METRIC ACCURACY RESULTS)")
        print("=" * 60)
        
        all_results.sort(key=lambda x: x['avg_error'])
        
        for rank, res in enumerate(all_results[:3], 1):
            print(f"RANK #{rank}: {res['filename']} (Origin: {res['source']})")
            print(f"   -> Average System Error: {res['avg_error']:.2f}%")
            print(f"   -> Measured Dimension:  {res['pred_long']:.2f}mm x {res['pred_short']:.2f}mm")
            print(f"   -> Error Profile:       Long Axis: {res['pct_long']:.2f}% | Short Axis: {res['pct_short']:.2f}%\n")
        print("=" * 60)


run_eval_pipeline(TRAIN_IMAGES_DIR, TEST_IMAGES_DIR, WEIGHTS_PATH, OUTPUT_DIR)

Launching evaluation loop over 11 total files (3 Train + Test)...

[TRAIN][IMG_20260609_020413_jpg.rf.hOithGS3S0D136Barq9T.jpg] Conf: 0.3736
   -> Long Dim (113mm): 196.93mm | Abs: 83.93mm | Pct: 74.27%
   -> Short Dim (62mm): 114.08mm | Abs: 52.08mm | Pct: 84.00%

[TRAIN][IMG_20260609_020416_jpg.rf.x06cv4dBrkMKb7y9Bgo0.jpg] Conf: 0.4004
   -> Long Dim (113mm): 189.98mm | Abs: 76.98mm | Pct: 68.13%
   -> Short Dim (62mm): 109.98mm | Abs: 47.98mm | Pct: 77.38%

[TRAIN][IMG_20260609_020422_jpg.rf.JTEQka9lqWvDeuE8UgBp.jpg] Target localization breach - skipping metrics.

[TEST][IMG_20260609_020459_jpg.rf.dsj17UarY3rbZsB4R1er.jpg] Conf: 0.1904
   -> Long Dim (113mm): 246.79mm | Abs: 133.79mm | Pct: 118.40%
   -> Short Dim (62mm): 124.88mm | Abs: 62.88mm | Pct: 101.42%

[TEST][IMG_20260609_020553_jpg.rf.2JLL6eceo4R00W15gtlv.jpg] Conf: 0.2810
   -> Long Dim (113mm): 98.25mm | Abs: 14.75mm | Pct: 13.05%
   -> Short Dim (62mm): 80.33mm | Abs: 18.33mm | Pct: 29.56%

[TEST][IMG_20260609_020616_jp

In [6]:
import os
import zipfile
from IPython.display import FileLink, display

# 1. Define your directory and zip file paths
folder_path = "/kaggle/working/outputfolder_revised"
zip_path = "Output_Results.zip"

# Check if the directory exists and contains images before zipping
if not os.path.exists(folder_path):
    print(f"❌ ERROR: The directory {folder_path} does not exist. Run your main pipeline script first!")
else:
    files_to_zip = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(files_to_zip)} processed images in {folder_path}.")

    # 2. Loop through the directory and compress all contents
    print("Compressing visualized measurement frames into a zip file...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                # Get full path of the image file
                file_path = os.path.join(root, file)
                # arcname ensures files don't carry the absolute Kaggle directory structure into the zip
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname=arcname)
                print(f" -> Added to archive: {arcname}")

    print("\n✅ Compression complete!")

    # 3. Generate the clickable download link for your local machine
    print("\n👇 Click the link below to download your visual polygon results:")
    display(FileLink(zip_path))

Found 11 processed images in /kaggle/working/outputfolder_revised.
Compressing visualized measurement frames into a zip file...
 -> Added to archive: IMG_20260609_020459_jpg.rf.dsj17UarY3rbZsB4R1er.jpg
 -> Added to archive: IMG_20260609_021032_jpg.rf.DYEQjnWM9BGJsHSt6bVE.jpg
 -> Added to archive: IMG_20260609_020416_jpg.rf.x06cv4dBrkMKb7y9Bgo0.jpg
 -> Added to archive: IMG_20260609_020553_jpg.rf.2JLL6eceo4R00W15gtlv.jpg
 -> Added to archive: IMG_20260609_020616_jpg.rf.2dmx73IxDmxBTVr04wMZ.jpg
 -> Added to archive: IMG_20260609_020943__01_jpg.rf.836fH1szQ9PbO61oOFep.jpg
 -> Added to archive: IMG_20260609_020943_jpg.rf.rdCgxq1EVKIpen5OLaeR.jpg
 -> Added to archive: IMG_20260609_020413_jpg.rf.hOithGS3S0D136Barq9T.jpg
 -> Added to archive: IMG_20260609_020946_jpg.rf.BRouITSHw35AfDSGMRAX.jpg
 -> Added to archive: IMG_20260609_021101_jpg.rf.vxVqR8bNtJZALsFjznxI.jpg
 -> Added to archive: IMG_20260609_020422_jpg.rf.JTEQka9lqWvDeuE8UgBp.jpg

✅ Compression complete!

👇 Click the link below to do

/kaggle/working/Output_Results.zip